# Document 2 — Neural Network Implementation
## Loan Default Risk Prediction System

**Student:** Narjiss Maimouni    
**Date:** May

---

### Introduction - Building a Multilayer Perceptron (MLP) From Scratch Using NumPy
In this document, we build a neural network from scratch using only NumPy, without using deep learning libraries such as TensorFlow or PyTorch.

The goal of this model is to predict whether a customer will default on a loan using the cleaned dataset prepared in Document 3.

The neural network used in this project is a Multilayer Perceptron (MLP) composed of:

An input layer with 11 features
One hidden layer using ReLU activation
One output layer using Sigmoid activation
Binary Cross Entropy as the loss function

The model will learn by:

Making predictions
Measuring prediction error
Updating weights using backpropagation
Repeating this process over multiple epochs

In [1]:
import numpy as np 
import pandas as pd

df = pd.read_csv('clean_dataset.csv')

# Separate features and label 
X = df.drop(columns=['Default']).values
Y = df['Default'].values

# Confirm
print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"Class distribution: {np.bincount(Y.astype(int))}")

X shape: (40026, 11)
Y shape: (40026,)
Class distribution: [30000 10026]


In [2]:
# Train/Test Split

## Why do we Split the Dataset?
# A machine learning model must be evaluated on data it has never seen before.
# If we train and test on the same data: . The model may memorize patterns
###                                      . Evaluation becomes unrealistic
###                                      . The model may fial in real-world situation
# To avoid this  :   . 70% of teh data is used for training
###                  . 30% is used for testing

In [3]:
# Shuffle indices
np.random.seed(42)

indices = np.random.permutation(len(X))

## Apply shuffle
X = X[indices]
Y = Y[indices]

# Split index
split_index = int(0.7 * len(X))

# Train/Test split
X_train = X[:split_index]
X_test = X[split_index:]

Y_train = Y[:split_index]
Y_test = Y[split_index:]

# Confirm shapes
print("Training set:")
print(X_train.shape, Y_train.shape)

print("\nTesting set:")
print(X_test.shape, Y_test.shape)

Training set:
(28018, 11) (28018,)

Testing set:
(12008, 11) (12008,)


In [4]:
# Why Was The Dataset Shuffled Again ?

## Even though we already shuffled the dataset in Document 2, we shuffle again before splitting to ensure :
####   . Random distribution of samples
####   . Fair separation between training and testing data
####   . Reduced risk of hidden ordering patterns

## This helps the model learn more reliably 

In [5]:
# Network Architecture 
input_size = 11
hidden_size = 16
output_size = 1

# Initialize weights
np.random.seed(42)

W1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros((1, output_size))

# Confirm shapes
print("W1 shape:", W1.shape)
print("b1 shape:", b1.shape)

print("W2 shape:", W2.shape)
print("b2 shape:", b2.shape)

W1 shape: (11, 16)
b1 shape: (1, 16)
W2 shape: (16, 1)
b2 shape: (1, 1)


In [6]:
## why use 16 Hidden Neurons ?
# we have selected a hidden layer size of 16 neurons . 
# This provides sufficient capacityy to capture non-linear interactions between financial features
# (such as the relationship between DebtRatio and MonthlyIncome) without risking overfitting. 



## Why We Initialize The Weights With Small Random Values?
# If all weights start with identical values:  . Every neuron learns the same thing
#                                              . The network becomes useless

# Using small random values helps neurons:     . Learn different patterns
#                                              . Specialize during training

# The multiplication by 0.01 prevents extremely large outputs at the beginning of training.

In [7]:
# ReLU activation
def relu(Z):
    return np.maximum(0, Z)

# ReLU derivative
def relu_derivative(Z):
    return Z > 0

# Sigmoid activation 
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

In [8]:
## Forward Propagation is th eprocess where: 
#      1. Input move through the  network
#      2. Neurons perform calculations
#      3. Activations are applied
#      4. Predictions are produces

## The information flows from: input layer --> hidden layer --> output layer


### FORWARD PROPAGATION EQUATIONS :
#     Hidden layer : Z1 = X.W1 + b1
#     Then activation: A1 = ReLU(Z1)
#     Output layer:  Z2 = A1.W2 + b2
#     Final prediction:  

In [9]:
# Forward Propagation function
def forward_propagation(X, W1, b1, W2, b2):
    
    # Hidden layer
    Z1 = np.dot(X, W1) + b1
    A1= relu(Z1)

    # output layer 
    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2)

    return Z1, A1, Z2, A2

In [10]:
## Binary Cross Entropy Loss
# why do we need a Loss Function? the loss function measures how wrong the model predictions are.
# A small loss: means predictions are close to real values.
# A largee loss: means predictions are wrong.
# The goal of training is to reduce the loss as much as possible.